In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load synthetic data ===
df = pd.read_csv("generated_data_Our_prompts_MIMIC.csv")

# === Ensure los_seconds is binary (already binarized as per your note)
df = df.dropna(subset=["los_seconds"])
df["los_seconds"] = df["los_seconds"].astype(int)

# === Extract features and labels
X = df.drop(columns=["los_seconds"])
y = df["los_seconds"]

# === Convert object/bool columns to numeric
for col in X.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = X[col].dropna().unique().tolist()
    if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
        X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    else:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Define models
models = {
    'pred_decision_tree': DecisionTreeClassifier(random_state=42),
    'pred_logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'pred_random_forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'pred_svm': SVC(probability=True, random_state=42),
    'pred_xgboost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Create a copy of test set for output
df_test_with_preds = X_test.copy()
df_test_with_preds["true_label"] = y_test.values

# === Train and predict
for name, model in models.items():
    model.fit(X_train, y_train)
    df_test_with_preds[name] = model.predict(X_test)

# === Save the output
df_test_with_preds.to_csv("synthetic_our_with_predictions.csv", index=False)
print("✅ Saved predictions to 'synthetic_test_with_predictions.csv'")


c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


✅ Saved predictions to 'synthetic_test_with_predictions.csv'


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from tqdm import tqdm

# === Load synthetic data (los_seconds is already binarized) ===
df_synth = pd.read_csv("generated_data_Our_prompts_MIMIC.csv")

# === Split synthetic data 80/20
df_train, df_test = train_test_split(df_synth, test_size=0.2, stratify=df_synth["los_seconds"], random_state=42)

# === Preserve original race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Extract X, y
X_train = df_train.drop(columns=["los_seconds"])
y_train = df_train["los_seconds"]
X_test = df_test.drop(columns=["los_seconds"])
y_test = df_test["los_seconds"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Define models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Header
print("\nFairness Metrics (Privileged: race='WHITE', Non-privileged: all others)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluation
for name in tqdm(models, desc="Evaluating models"):
    model = models[name]
    print(f"\n▶ Evaluating: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # Convert to NumPy for safe indexing
    y_pred_np = np.array(y_pred)
    y_test_np = np.array(y_test)
    race_test_np = np.array(race_test)

    # Define groups
    group_priv = race_test_np == white_code
    group_unpriv = race_test_np != white_code

    # === ERD
    err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
    err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
    tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        y_prob_np = np.array(y_prob)
        try:
            auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob_np[group_priv])
            auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob_np[group_unpriv])
            abroca = abs(auc_priv - auc_unpriv)
        except:
            abroca = np.nan

    # === Fairness Score
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race='WHITE', Non-privileged: all others)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------


Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]


▶ Evaluating: Decision Tree
Decision Tree        0.0011     0.0444     -0.0519    0.9675    

▶ Evaluating: Logistic Regression
Logistic Regression  0.0764     0.0114     0.0000     0.9707    

▶ Evaluating: Random Forest


Evaluating models:  60%|██████    | 3/5 [00:00<00:00,  6.67it/s]

Random Forest        0.0461     0.0084     -0.0038    0.9806    

▶ Evaluating: SVM


Evaluating models: 100%|██████████| 5/5 [00:01<00:00,  4.43it/s]

SVM                  0.0699     0.0174     -0.0068    0.9686    

▶ Evaluating: XGBoost
XGBoost              0.0267     0.0120     -0.0181    0.9811    


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from tqdm import tqdm

# === Load synthetic data (los_seconds is already binarized) ===
df_synth = pd.read_csv("generated_data_CLLM_prompt_Mimic.csv")

# === Split synthetic data 80/20
df_train, df_test = train_test_split(df_synth, test_size=0.2, stratify=df_synth["los_seconds"], random_state=42)

# === Preserve original race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Extract X, y
X_train = df_train.drop(columns=["los_seconds"])
y_train = df_train["los_seconds"]
X_test = df_test.drop(columns=["los_seconds"])
y_test = df_test["los_seconds"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Define models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Header
print("\nFairness Metrics (Privileged: race='WHITE', Non-privileged: all others)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluation
for name in tqdm(models, desc="Evaluating models"):
    model = models[name]
    print(f"\n▶ Evaluating: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # Convert to NumPy for safe indexing
    y_pred_np = np.array(y_pred)
    y_test_np = np.array(y_test)
    race_test_np = np.array(race_test)

    # Define groups
    group_priv = race_test_np == white_code
    group_unpriv = race_test_np != white_code

    # === ERD
    err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
    err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
    tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        y_prob_np = np.array(y_prob)
        try:
            auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob_np[group_priv])
            auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob_np[group_unpriv])
            abroca = abs(auc_priv - auc_unpriv)
        except:
            abroca = np.nan

    # === Fairness Score
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race='WHITE', Non-privileged: all others)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------


Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]


▶ Evaluating: Decision Tree
Decision Tree        0.0739     0.0630     -0.0295    0.9445    

▶ Evaluating: Logistic Regression
Logistic Regression  0.0282     -0.0058    0.0000     0.9887    

▶ Evaluating: Random Forest


Evaluating models:  60%|██████    | 3/5 [00:00<00:00,  6.51it/s]

Random Forest        0.0287     -0.0295    0.0405     0.9671    

▶ Evaluating: SVM


Evaluating models: 100%|██████████| 5/5 [00:02<00:00,  2.35it/s]

SVM                  0.0596     -0.0085    0.0000     0.9773    

▶ Evaluating: XGBoost
XGBoost              0.0049     -0.0248    0.0183     0.9840    


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from tqdm import tqdm

# === Load synthetic data (los_seconds is already binarized) ===
df_synth = pd.read_csv("mimic_synthetic_data_3400_samples_DECAF.csv")

# === Binarize los_seconds: 1 if >= 345600, else 0
df_synth["label"] = (df_synth["los_seconds"] >= 345600).astype(int)

# === Split synthetic data 80/20 based on the new label
df_train, df_test = train_test_split(df_synth, test_size=0.2, stratify=df_synth["label"], random_state=42)

# === Preserve original race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Extract X, y
X_train = df_train.drop(columns=["los_seconds", "label"])
y_train = df_train["label"]
X_test = df_test.drop(columns=["los_seconds", "label"])
y_test = df_test["label"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Define models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Header
print("\nFairness Metrics (Privileged: race='WHITE', Non-privileged: all others)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluation
for name in tqdm(models, desc="Evaluating models"):
    model = models[name]
    print(f"\n▶ Evaluating: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # Convert to NumPy for safe indexing
    y_pred_np = np.array(y_pred)
    y_test_np = np.array(y_test)
    race_test_np = np.array(race_test)

    # Define groups
    group_priv = race_test_np == white_code
    group_unpriv = race_test_np != white_code

    # === ERD
    err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
    err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
    tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        y_prob_np = np.array(y_prob)
        try:
            auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob_np[group_priv])
            auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob_np[group_unpriv])
            abroca = abs(auc_priv - auc_unpriv)
        except:
            abroca = np.nan

    # === Fairness Score
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race='WHITE', Non-privileged: all others)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------


Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]


▶ Evaluating: Decision Tree
Decision Tree        0.1228     0.1052     -0.1897    0.8608    

▶ Evaluating: Logistic Regression
Logistic Regression  0.0046     0.0410     -0.0751    0.9598    

▶ Evaluating: Random Forest


Evaluating models:  60%|██████    | 3/5 [00:00<00:00,  6.37it/s]

Random Forest        0.0596     0.0645     -0.1175    0.9195    

▶ Evaluating: SVM


Evaluating models: 100%|██████████| 5/5 [00:02<00:00,  2.40it/s]

SVM                  0.0150     0.0003     0.0000     0.9949    

▶ Evaluating: XGBoost
XGBoost              0.0627     0.1417     -0.2134    0.8607    


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier

# === Load full dataset ===
df = pd.read_csv("Real_MIMIC.csv")

# === Binarize los_seconds: 1 if >= 345600, else 0
df["label"] = (df["los_seconds"] >= 345600).astype(int)

# === Split full data 80/20
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

# === Preserve race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Prepare features and labels
X_train = df_train.drop(columns=["los_seconds", "label"])
y_train = df_train["label"]
X_test = df_test.drop(columns=["los_seconds", "label"])
y_test = df_test["label"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# === Convert to NumPy for indexing
y_pred_np = np.array(y_pred)
y_test_np = np.array(y_test)
race_test_np = np.array(race_test)

# === Define privileged/unprivileged groups
group_priv = race_test_np == white_code
group_unpriv = race_test_np != white_code

# === ERD
err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
erd = err_unpriv - err_priv

# === TPRD
def tpr(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0

tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
tprd = tpr_unpriv - tpr_priv

# === ABROCA
def compute_roc_auc(y_true, y_score):
    fpr, tpr_vals, _ = roc_curve(y_true, y_score)
    return auc(fpr, tpr_vals)

abroca = np.nan
try:
    auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob[group_priv])
    auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob[group_unpriv])
    abroca = abs(auc_priv - auc_unpriv)
except:
    abroca = np.nan

# === Fairness Score
fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3 if not np.isnan(abroca) else float("nan")

# === Print result
print("\nFairness Metrics (XGBoost on Full Data)")
print("--------------------------------------------------------------------------")
print(f"{'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")
print(f"{abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (XGBoost on Full Data)
--------------------------------------------------------------------------
ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
0.0114     -0.0205    -0.0413    0.9756    


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier

# === Load real and synthetic data ===
df_real = pd.read_csv("Real_MIMIC.csv")
df_synth = pd.read_csv("mimic_synthetic_data_3400_samples_DECAF.csv")

# === Binarize los_seconds: 1 if >= 345600, else 0
df_real["label"] = (df_real["los_seconds"] >= 345600).astype(int)
df_synth["label"] = (df_synth["los_seconds"] >= 345600).astype(int)

# === Split real data 80/20 for test only
_, df_test = train_test_split(df_real, test_size=0.2, stratify=df_real["label"], random_state=42)

# === Combine real + synthetic data for training
df_train = pd.concat([df_real, df_synth], ignore_index=True)

# === Preserve race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Prepare X/y
X_train = df_train.drop(columns=["los_seconds", "label"])
y_train = df_train["label"]
X_test = df_test.drop(columns=["los_seconds", "label"])
y_test = df_test["label"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# === Convert to NumPy for safe indexing
y_pred_np = np.array(y_pred)
y_test_np = np.array(y_test)
race_test_np = np.array(race_test)

# === Define groups
group_priv = race_test_np == white_code
group_unpriv = race_test_np != white_code

# === ERD
err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
erd = err_unpriv - err_priv

# === TPRD
def tpr(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0

tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
tprd = tpr_unpriv - tpr_priv

# === ABROCA
def compute_roc_auc(y_true, y_score):
    fpr, tpr_vals, _ = roc_curve(y_true, y_score)
    return auc(fpr, tpr_vals)

abroca = np.nan
try:
    auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob[group_priv])
    auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob[group_unpriv])
    abroca = abs(auc_priv - auc_unpriv)
except:
    abroca = np.nan

# === Fairness Score
fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3 if not np.isnan(abroca) else float("nan")

# === Print result
print("\nFairness Metrics (XGBoost using Real+Synthetic train, Real test)")
print("--------------------------------------------------------------------------")
print(f"{'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")
print(f"{abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (XGBoost using Real+Synthetic train, Real test)
--------------------------------------------------------------------------
ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
0.0198     -0.0331    -0.0352    0.9706    


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier

# === Load real and synthetic data ===
df_real = pd.read_csv("Real_MIMIC.csv")
df_synth = pd.read_csv("generated_data_CLLM_prompt_Mimic.csv")

# === Binarize los_seconds: 1 if >= 345600, else 0
df_real["label"] = (df_real["los_seconds"] >= 345600).astype(int)
df_synth["label"] = (df_synth["los_seconds"]).astype(int)

# === Split real data 80/20 for test only
_, df_test = train_test_split(df_real, test_size=0.2, stratify=df_real["label"], random_state=42)

# === Combine real + synthetic data for training
df_train = pd.concat([df_real, df_synth], ignore_index=True)

# === Preserve race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Prepare X/y
X_train = df_train.drop(columns=["los_seconds", "label"])
y_train = df_train["label"]
X_test = df_test.drop(columns=["los_seconds", "label"])
y_test = df_test["label"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# === Convert to NumPy for safe indexing
y_pred_np = np.array(y_pred)
y_test_np = np.array(y_test)
race_test_np = np.array(race_test)

# === Define groups
group_priv = race_test_np == white_code
group_unpriv = race_test_np != white_code

# === ERD
err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
erd = err_unpriv - err_priv

# === TPRD
def tpr(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0

tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
tprd = tpr_unpriv - tpr_priv

# === ABROCA
def compute_roc_auc(y_true, y_score):
    fpr, tpr_vals, _ = roc_curve(y_true, y_score)
    return auc(fpr, tpr_vals)

abroca = np.nan
try:
    auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob[group_priv])
    auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob[group_unpriv])
    abroca = abs(auc_priv - auc_unpriv)
except:
    abroca = np.nan

# === Fairness Score
fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3 if not np.isnan(abroca) else float("nan")

# === Print result
print("\nFairness Metrics (XGBoost using Real+Synthetic train, Real test)")
print("--------------------------------------------------------------------------")
print(f"{'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")
print(f"{abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (XGBoost using Real+Synthetic train, Real test)
--------------------------------------------------------------------------
ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
0.0182     -0.0397    0.0237     0.9728    


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier

# === Load real and synthetic data ===
df_real = pd.read_csv("Real_MIMIC.csv")
df_synth = pd.read_csv("generated_data_Our_prompts_MIMIC.csv")

# === Binarize los_seconds: 1 if >= 345600, else 0
df_real["label"] = (df_real["los_seconds"] >= 345600).astype(int)
df_synth["label"] = (df_synth["los_seconds"]).astype(int)

# === Split real data 80/20 for test only
_, df_test = train_test_split(df_real, test_size=0.2, stratify=df_real["label"], random_state=42)

# === Combine real + synthetic data for training
df_train = pd.concat([df_real, df_synth], ignore_index=True)

# === Preserve race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Prepare X/y
X_train = df_train.drop(columns=["los_seconds", "label"])
y_train = df_train["label"]
X_test = df_test.drop(columns=["los_seconds", "label"])
y_test = df_test["label"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# === Convert to NumPy for safe indexing
y_pred_np = np.array(y_pred)
y_test_np = np.array(y_test)
race_test_np = np.array(race_test)

# === Define groups
group_priv = race_test_np == white_code
group_unpriv = race_test_np != white_code

# === ERD
err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
erd = err_unpriv - err_priv

# === TPRD
def tpr(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0

tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
tprd = tpr_unpriv - tpr_priv

# === ABROCA
def compute_roc_auc(y_true, y_score):
    fpr, tpr_vals, _ = roc_curve(y_true, y_score)
    return auc(fpr, tpr_vals)

abroca = np.nan
try:
    auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob[group_priv])
    auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob[group_unpriv])
    abroca = abs(auc_priv - auc_unpriv)
except:
    abroca = np.nan

# === Fairness Score
fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3 if not np.isnan(abroca) else float("nan")

# === Print result
print("\nFairness Metrics (XGBoost using Real+Synthetic train, Real test)")
print("--------------------------------------------------------------------------")
print(f"{'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")
print(f"{abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (XGBoost using Real+Synthetic train, Real test)
--------------------------------------------------------------------------
ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
0.0147     0.0191     -0.0001    0.9887    


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier

# === Load real and synthetic datasets ===
df_real = pd.read_csv("Real_MIMIC.csv")
df_synth = pd.read_csv("mimic_synthetic_data_3400_samples_DECAF.csv")

# === Binarize los_seconds: 1 if >= 345600, else 0
df_real["label"] = (df_real["los_seconds"] >= 345600).astype(int)
df_synth["label"] = (df_synth["los_seconds"] >= 345600).astype(int)

# === Combine real + synthetic data
df_all = pd.concat([df_real, df_synth], ignore_index=True)

# === Train/test split on full combined dataset
df_train, df_test = train_test_split(df_all, test_size=0.2, stratify=df_all["label"], random_state=42)

# === Preserve race for fairness analysis
df_train["race_original"] = df_train["race"]
df_test["race_original"] = df_test["race"]

# === Prepare X, y
X_train = df_train.drop(columns=["los_seconds", "label"])
y_train = df_train["label"]
X_test = df_test.drop(columns=["los_seconds", "label"])
y_test = df_test["label"]

# === Encode object/bool columns
for X in [X_train, X_test]:
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        unique_vals = X[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute missing values
X_train = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(SimpleImputer(strategy='most_frequent').fit_transform(X_test), columns=X_test.columns)

# === Encode race for fairness comparison
race_encoder = LabelEncoder()
race_encoder.fit(pd.concat([df_train["race_original"], df_test["race_original"]], ignore_index=True))
X_train["race"] = race_encoder.transform(df_train["race_original"].astype(str))
X_test["race"] = race_encoder.transform(df_test["race_original"].astype(str))
white_code = race_encoder.transform(["WHITE"])[0]
race_test = X_test["race"]

# === Train XGBoost
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# === Convert to NumPy for safe indexing
y_pred_np = np.array(y_pred)
y_test_np = np.array(y_test)
race_test_np = np.array(race_test)

# === Define groups
group_priv = race_test_np == white_code
group_unpriv = race_test_np != white_code

# === ERD
err_priv = np.mean(y_pred_np[group_priv] != y_test_np[group_priv])
err_unpriv = np.mean(y_pred_np[group_unpriv] != y_test_np[group_unpriv])
erd = err_unpriv - err_priv

# === TPRD
def tpr(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0

tpr_priv = tpr(y_test_np[group_priv], y_pred_np[group_priv])
tpr_unpriv = tpr(y_test_np[group_unpriv], y_pred_np[group_unpriv])
tprd = tpr_unpriv - tpr_priv

# === ABROCA
def compute_roc_auc(y_true, y_score):
    fpr, tpr_vals, _ = roc_curve(y_true, y_score)
    return auc(fpr, tpr_vals)

abroca = np.nan
try:
    auc_priv = compute_roc_auc(y_test_np[group_priv], y_prob[group_priv])
    auc_unpriv = compute_roc_auc(y_test_np[group_unpriv], y_prob[group_unpriv])
    abroca = abs(auc_priv - auc_unpriv)
except:
    abroca = np.nan

# === Fairness Score
fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3 if not np.isnan(abroca) else float("nan")

# === Print result
print("\nFairness Metrics (XGBoost on Real+Synthetic for both train and test)")
print("--------------------------------------------------------------------------")
print(f"{'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")
print(f"{abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (XGBoost on Real+Synthetic for both train and test)
--------------------------------------------------------------------------
ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
0.0146     -0.0224    -0.0593    0.9679    
